#### TECHNICAL GUIDES / DATA SCIENCE & AI

# SQL y bases de datos

Carlos A. Ugarte

#### TECHNICAL GUIDES / RESEARCH - EDUCATION - CONSULTING

SQLite en memoria, datos ficticios; no modifica bases externas. Las instrucciones MySQL se muestran como referencia, no se ejecutan. Los resultados decimales se formatean a cuatro posiciones.

In [1]:
import sqlite3 as db, pandas as pd, json, tempfile
from pathlib import Path
from IPython.display import display
pd.options.display.float_format = '{:.4f}'.format
c=db.connect(':memory:'); c.execute('PRAGMA foreign_keys=ON')
def q(sql, params=()):
    display(pd.read_sql_query(sql,c,params=params))
def run(sql):
    c.executescript(sql)
print('SQLite:', db.sqlite_version)


SQLite: 3.42.0


## I. Guía integral de bases de datos

Una base de datos es una colección estructurada de datos almacenados electrónicamente para gestionar y recuperar información de manera eficiente. Se utiliza en comercio electrónico, banca, salud, educación y redes sociales.

## 1.1–1.2 Definición y necesidad

El almacenamiento organizado facilita manejar grandes volúmenes. La integridad reduce duplicaciones y errores; la escalabilidad permite crecer; la concurrencia admite varios usuarios; la seguridad protege contra accesos no autorizados. Estas propiedades dependen del diseño y del motor, no son automáticas.

In [2]:
run("CREATE TABLE ejemplo(id INTEGER PRIMARY KEY, nombre TEXT NOT NULL); INSERT INTO ejemplo VALUES(1,'Ana'),(2,'Luis');")
q('SELECT * FROM ejemplo')

,id,nombre
0,1,Ana
1,2,Luis


## 1.3 Tipos de bases de datos

Relacionales: tablas y claves (MySQL, PostgreSQL, OracleDB). NoSQL: documentos, clave-valor, familias de columnas y grafos (MongoDB, Cassandra). Jerárquicas: árbol padre-hijo (IBM IMS). De red: múltiples relaciones. Orientadas a objetos: atributos y métodos. Nota: NoSQL también admite datos estructurados; el modelo de red histórico no es idéntico a un motor de grafos moderno.

In [3]:
documentos=[{'id':1,'telefonos':['123','456']},{'id':2,'telefonos':['789']}]
clave_valor={x['id']:x for x in documentos}
familias={1:{'contacto':{'tel':'123'},'ventas':{'total':100}}}
arbol={'Dirección':{'Ventas':{},'TI':{}}}
grafo={'Ana':['Luis','Eva'],'Luis':['Eva']}
class Cliente:
    def __init__(self,nombre): self.nombre=nombre
    def saludo(self): return 'Hola '+self.nombre
print(json.dumps(documentos,ensure_ascii=False), clave_valor[1], familias, arbol, grafo, Cliente('Ana').saludo())
# Representaciones didácticas, no motores NoSQL reales.

[{"id": 1, "telefonos": ["123", "456"]}, {"id": 2, "telefonos": ["789"]}] {'id': 1, 'telefonos': ['123', '456']} {1: {'contacto': {'tel': '123'}, 'ventas': {'total': 100}}} {'Dirección': {'Ventas': {}, 'TI': {}}} {'Ana': ['Luis', 'Eva'], 'Luis': ['Eva']} Hola Ana


## 1.4 Componentes principales

Tablas: filas y columnas. Esquema: definición de estructura, relaciones y restricciones. Clave primaria: identifica un registro; foránea: referencia otra tabla; compuesta: combina columnas. Índices aceleran búsquedas; consultas manipulan y recuperan datos; vistas son tablas virtuales; restricciones como UNIQUE y NOT NULL preservan integridad.

In [4]:
run("""CREATE TABLE customers(id INTEGER PRIMARY KEY,name TEXT NOT NULL UNIQUE);
CREATE TABLE products(id INTEGER PRIMARY KEY,name TEXT,price_cents INTEGER CHECK(price_cents>=0));
CREATE TABLE orders(order_id INTEGER PRIMARY KEY,customer_id INTEGER REFERENCES customers(id),amount_cents INTEGER NOT NULL CHECK(amount_cents>=0));
CREATE TABLE details(order_id INTEGER REFERENCES orders(order_id),product_id INTEGER REFERENCES products(id),qty INTEGER CHECK(qty>0),PRIMARY KEY(order_id,product_id));
INSERT INTO customers VALUES(1,'Ana'),(2,'Luis'),(3,'Eva');
INSERT INTO products VALUES(1,'A',1000),(2,'B',2500);
INSERT INTO orders VALUES(1,1,1000),(2,1,5000),(3,2,2500);
INSERT INTO details VALUES(1,1,1),(2,2,2),(3,2,1);""")
q('PRAGMA table_info(details)'); q('PRAGMA foreign_key_list(details)')

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,INTEGER,0,None,1
1,1,product_id,INTEGER,0,None,2
2,2,qty,INTEGER,0,None,0


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,products,product_id,id,NO ACTION,NO ACTION,NONE
1,1,0,orders,order_id,order_id,NO ACTION,NO ACTION,NONE


## 1.5.1 Normalización

La normalización divide tablas para reducir redundancia. 1FN: valores atómicos, sin grupos repetidos; Juan con dos teléfonos se transforma en dos filas. 2FN: elimina dependencias parciales de claves compuestas; nombre de producto depende del producto, no del pedido. 3FN: elimina dependencias transitivas; nombre de departamento depende del departamento, no del estudiante. BCNF: todo determinante de una dependencia funcional no trivial debe ser superclave (precisión respecto del original, que dice clave candidata). 4FN: elimina dependencias multivaluadas no triviales cuyo determinante no es superclave; separar hobbies y materias independientes. 5FN: elimina redundancia por dependencias de reunión no implicadas por claves; la descomposición debe ser sin pérdida. Desnormalización: combinar datos para favorecer lecturas, con riesgo de inconsistencias.

In [5]:
telefonos=pd.DataFrame([('Juan',['123','456'])],columns=['nombre','telefono']); display(telefonos.explode('telefono'))
q('SELECT d.order_id,d.product_id,p.name,d.qty FROM details d JOIN products p ON p.id=d.product_id')
run("""CREATE TABLE departments(id INTEGER PRIMARY KEY,name TEXT); INSERT INTO departments VALUES(1,'TI'),(2,'Ventas'); CREATE TABLE students(id INTEGER PRIMARY KEY,dept_id INTEGER REFERENCES departments(id)); INSERT INTO students VALUES(1,1),(2,1),(3,2);""")
q('SELECT s.id,d.name FROM students s JOIN departments d ON s.dept_id=d.id')
# BCNF: docente determina materia; se separa esa dependencia.
clases=pd.DataFrame([('Ana','SQL','Eva'),('Luis','SQL','Eva'),('Ana','Python','Leo')],columns=['alumno','materia','docente'])
display(clases[['docente','materia']].drop_duplicates()); display(clases[['alumno','docente']].drop_duplicates())
# 4FN: dos conjuntos independientes, sin producto cartesiano almacenado.
hobbies=pd.DataFrame({'alumno':['Ana']*2,'hobby':['lectura','piano']}); materias=pd.DataFrame({'alumno':['Ana']*2,'materia':['SQL','Python']}); display(hobbies.merge(materias,on='alumno'))
# 5FN: ejemplo donde las tres compatibilidades binarias definen los triples válidos.
sp=pd.DataFrame([('A','X'),('A','Y'),('B','X')],columns=['proveedor','producto']); sj=pd.DataFrame([('A','J'),('B','J')],columns=['proveedor','proyecto']); pj=pd.DataFrame([('X','J')],columns=['producto','proyecto']); display(sp.merge(sj,on='proveedor').merge(pj,on=['producto','proyecto']))
q('SELECT o.order_id,c.name,o.amount_cents/100.0 AS importe FROM orders o JOIN customers c ON c.id=o.customer_id')

,nombre,telefono
0,Juan,123
0,Juan,456


,order_id,product_id,name,qty
0,1,1,A,1
1,2,2,B,2
2,3,2,B,1


,id,name
0,1,TI
1,2,TI
2,3,Ventas


,docente,materia
0,Eva,SQL
2,Leo,Python


,alumno,docente
0,Ana,Eva
1,Luis,Eva
2,Ana,Leo


,alumno,hobby,materia
0,Ana,lectura,SQL
1,Ana,lectura,Python
2,Ana,piano,SQL
3,Ana,piano,Python


,proveedor,producto,proyecto
0,A,X,J
1,B,X,J


,order_id,name,importe
0,1,Ana,10.0000
1,2,Ana,50.0000
2,3,Luis,25.0000


## 1.5.2–1.5.3 Relaciones y modelo ER

Uno a uno: persona y pasaporte (clave foránea única). Uno a muchos: cliente y pedidos. Muchos a muchos: estudiantes y cursos mediante tabla puente. El modelo entidad-relación representa entidades, atributos y relaciones; un diagrama ER muestra su estructura.

In [6]:
run("""CREATE TABLE passports(person_id INTEGER PRIMARY KEY REFERENCES customers(id),number TEXT UNIQUE); INSERT INTO passports VALUES(1,'P001'); CREATE TABLE courses(id INTEGER PRIMARY KEY,name TEXT); INSERT INTO courses VALUES(1,'SQL'),(2,'Python'); CREATE TABLE enrollment(student_id INTEGER REFERENCES students(id),course_id INTEGER REFERENCES courses(id),PRIMARY KEY(student_id,course_id)); INSERT INTO enrollment VALUES(1,1),(1,2),(2,1);""")
q('SELECT * FROM enrollment'); q('SELECT customer_id,COUNT(*) AS pedidos FROM orders GROUP BY customer_id')
print('customers 1 ── N orders 1 ── N details N ── 1 products; students N ── M courses (enrollment)')

,student_id,course_id
0,1,1
1,1,2
2,2,1


,customer_id,pedidos
0,1,2
1,2,1


customers 1 ── N orders 1 ── N details N ── 1 products; students N ── M courses (enrollment)


## 1.5.4–1.5.5 Integridad y ACID

Integridad de entidad: identificadores únicos; referencial: claves foráneas válidas; de dominio: tipos y restricciones válidos. ACID: atomicidad (todo o nada), consistencia (preservar reglas), aislamiento (visibilidad controlada entre transacciones) y durabilidad (persistencia tras confirmar, según configuración).

In [8]:
import sqlite3 as db
import tempfile

from pathlib import Path
from contextlib import closing

# 1. Pruebas de restricciones de integridad
for sql in [
    "INSERT INTO customers VALUES(1,'Otro')",
    "INSERT INTO orders VALUES(9,99,100)",
    "INSERT INTO orders VALUES(9,1,-100)"
]:
    try:
        c.execute(sql)
        c.commit()

    except db.IntegrityError as e:
        c.rollback()
        print("Rechazado:", e)


# 2. Pruebas de aislamiento y persistencia
with tempfile.TemporaryDirectory() as tmp:

    path = Path(tmp) / "acid.db"

    with closing(db.connect(path)) as a, \
         closing(db.connect(path)) as b:

        # Crear tabla
        a.execute("CREATE TABLE t(x INTEGER)")
        a.commit()

        # Insertar registro sin confirmar la transacción
        a.execute("INSERT INTO t VALUES(1)")

        # Verificar aislamiento entre conexiones
        assert b.execute(
            "SELECT COUNT(*) FROM t"
        ).fetchone()[0] == 0

        print("OK: Aislamiento verificado.")

        # Confirmar transacción
        a.commit()

        # Verificar visibilidad después del commit
        assert b.execute(
            "SELECT COUNT(*) FROM t"
        ).fetchone()[0] == 1

        print("OK: Transacción confirmada.")

    # Ambas conexiones se encuentran cerradas

    # Reabrir la base de datos
    with closing(db.connect(path)) as reopened:

        assert reopened.execute(
            "SELECT x FROM t"
        ).fetchone()[0] == 1

        print("OK: Persistencia verificada.")

    # Todas las conexiones están cerradas.
    # Windows puede eliminar el directorio temporal.

print("\nPruebas de aislamiento y reapertura correctas.")
print("Estas pruebas no simulan un fallo eléctrico.")

Rechazado: UNIQUE constraint failed: customers.id
Rechazado: FOREIGN KEY constraint failed
Rechazado: CHECK constraint failed: amount_cents>=0
OK: Aislamiento verificado.
OK: Transacción confirmada.
OK: Persistencia verificada.

Pruebas de aislamiento y reapertura correctas.
Estas pruebas no simulan un fallo eléctrico.


## 1.6 Sistemas gestores populares

Relacionales: MySQL, PostgreSQL, OracleDB, SQL Server. NoSQL: MongoDB (documentos), Redis (clave-valor), Cassandra (familias de columnas). Nube: Amazon RDS, Google Firestore, Azure SQL Database. Nota: RDS es un servicio gestionado de motores relacionales; Firestore es documental.

In [9]:
display(pd.DataFrame([('SQLite','relacional','local'),('PostgreSQL','relacional','servidor'),('MongoDB','documentos','servidor'),('Redis','clave-valor','servidor'),('Cassandra','columnas','distribuido'),('Firestore','documentos','nube')],columns=['motor','modelo','entorno']))

,motor,modelo,entorno
0,SQLite,relacional,local
1,PostgreSQL,relacional,servidor
2,MongoDB,documentos,servidor
3,Redis,clave-valor,servidor
4,Cassandra,columnas,distribuido
5,Firestore,documentos,nube


## II. SQL básico

SQL (Structured Query Language) es un lenguaje declarativo estandarizado para consultar, insertar, actualizar, eliminar y definir datos relacionales. Se pronuncia S-Q-L o Sequel.

## 2.1–2.3 Sintaxis y clasificación

Las palabras reservadas no distinguen mayúsculas y minúsculas; los valores de texto sí pueden distinguirlas según colación. El punto y coma delimita instrucciones. DDL: CREATE, ALTER, DROP, TRUNCATE. DML: INSERT, UPDATE, DELETE, SELECT (también llamado DQL). DCL: GRANT, REVOKE. TCL: COMMIT, ROLLBACK, SAVEPOINT. CRUD: crear, leer, actualizar y eliminar registros. TRUNCATE y su registro transaccional dependen del motor.

In [10]:
run("CREATE TABLE demo(id INTEGER PRIMARY KEY); ALTER TABLE demo ADD COLUMN texto TEXT; INSERT INTO demo VALUES(1,'a'); UPDATE demo SET texto='b' WHERE id=1;")
q('SeLeCt * FROM demo'); run('DELETE FROM demo; DROP TABLE demo;')
# MySQL: TRUNCATE TABLE demo; GRANT SELECT ON my_database.* TO lector; REVOKE SELECT ON my_database.* FROM lector;
# SQLite no tiene GRANT/REVOKE: control de archivo y aplicación. Ejemplo de autorización local:
def autorizar(rol,operacion): return operacion in {'lector':{'SELECT'},'editor':{'SELECT','INSERT','UPDATE','DELETE'}}.get(rol,set())
assert autorizar('lector','SELECT') and not autorizar('lector','DELETE')

,id,texto
0,1,b


## 2.4 Comandos básicos

En MySQL: CREATE DATABASE my_database; USE my_database; CREATE TABLE users (id INT PRIMARY KEY AUTO_INCREMENT, name VARCHAR(50) NOT NULL, email VARCHAR(100) UNIQUE, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP); INSERT, SELECT, UPDATE, DELETE y DROP TABLE. En SQLite la conexión crea o abre la base, INTEGER PRIMARY KEY genera identificadores y no existe USE. Se usan parámetros para evitar inyección SQL.

In [11]:
run("""CREATE TABLE users(id INTEGER PRIMARY KEY,name TEXT NOT NULL,email TEXT UNIQUE,created_at TEXT DEFAULT CURRENT_TIMESTAMP,status TEXT NOT NULL DEFAULT 'active' CHECK(status IN ('active','inactive'))); INSERT INTO users(name,email) VALUES('John Doe','john@example.com'),('Jane','jane@example.com'),('Eva','eva@example.com');""")
c.execute('UPDATE users SET email=? WHERE id=?',('john.doe@example.com',1)); c.commit(); q('SELECT * FROM users'); c.execute('INSERT INTO users(name) VALUES(?)',('Temporal',)); c.execute('DELETE FROM users WHERE name=?',('Temporal',)); c.commit()

,id,name,email,created_at,status
0,1,John Doe,john.doe@example.com,2026-11-27 02:32:17,active
1,2,Jane,jane@example.com,2026-11-27 02:32:17,active
2,3,Eva,eva@example.com,2026-11-27 02:32:17,active


## 2.5 Restricciones

NOT NULL impide nulos; UNIQUE impide duplicados según semántica del motor (puede permitir varios NULL); PRIMARY KEY identifica filas; FOREIGN KEY referencia otra tabla; CHECK exige condiciones; DEFAULT asigna valor cuando se omite la columna. En SQLite CHECK admite NULL salvo NOT NULL adicional.

In [12]:
for sql in ["INSERT INTO users(name) VALUES(NULL)","INSERT INTO users(name,email) VALUES('X','jane@example.com')","INSERT INTO users(name,status) VALUES('X','invalid')"]:
    try: c.execute(sql); c.commit()
    except db.IntegrityError as e: c.rollback(); print(e)
q('SELECT name,status FROM users')

NOT NULL constraint failed: users.name
UNIQUE constraint failed: users.email
CHECK constraint failed: status IN ('active','inactive')


,name,status
0,John Doe,active
1,Jane,active
2,Eva,active


## 2.6–2.9 Filtros, operadores, orden y límite

WHERE selecciona filas. Comparaciones =, !=, <, >, <=, >=; AND, OR, NOT; LIKE usa % para cero o más caracteres y _ para uno; IN compara con una lista; BETWEEN incluye extremos. ORDER BY ASC/DESC ordena; LIMIT limita filas. Nota: usar IS NULL para nulos y ordenar con desempate para resultados reproducibles.

In [13]:
q("SELECT * FROM users WHERE email='john.doe@example.com'")

,id,name,email,created_at,status
0,1,John Doe,john.doe@example.com,2026-11-27 02:32:17,active


In [14]:
q("SELECT * FROM users WHERE (id>=1 AND id<=2) OR NOT status='active'")

,id,name,email,created_at,status
0,1,John Doe,john.doe@example.com,2026-11-27 02:32:17,active
1,2,Jane,jane@example.com,2026-11-27 02:32:17,active


In [15]:
q('SELECT * FROM users WHERE id!=2 AND id>0 AND id<4')

,id,name,email,created_at,status
0,1,John Doe,john.doe@example.com,2026-11-27 02:32:17,active
1,3,Eva,eva@example.com,2026-11-27 02:32:17,active


In [16]:
q("SELECT * FROM users WHERE name LIKE 'J%' OR name LIKE '_va'")

,id,name,email,created_at,status
0,1,John Doe,john.doe@example.com,2026-11-27 02:32:17,active
1,2,Jane,jane@example.com,2026-11-27 02:32:17,active
2,3,Eva,eva@example.com,2026-11-27 02:32:17,active


In [17]:
q('SELECT * FROM users WHERE id IN (1,3) AND id BETWEEN 1 AND 10')

,id,name,email,created_at,status
0,1,John Doe,john.doe@example.com,2026-11-27 02:32:17,active
1,3,Eva,eva@example.com,2026-11-27 02:32:17,active


In [18]:
q('SELECT * FROM users ORDER BY name ASC,id DESC LIMIT 2')

,id,name,email,created_at,status
0,3,Eva,eva@example.com,2026-11-27 02:32:17,active
1,2,Jane,jane@example.com,2026-11-27 02:32:17,active


## 2.10–2.11 Agregaciones y agrupación

COUNT cuenta filas o valores no nulos; SUM suma, AVG promedia, MAX y MIN identifican extremos. GROUP BY agrupa por columnas. Ampliación: HAVING filtra grupos; distinguir COUNT(*) de COUNT(columna).

In [19]:
run("""CREATE TABLE employees(id INTEGER PRIMARY KEY,name TEXT,department TEXT,department_id INTEGER,salary REAL,manager_id INTEGER REFERENCES employees(id)); INSERT INTO employees VALUES(1,'Eva','TI',101,9000,NULL),(2,'Ana','TI',101,6000,1),(3,'Luis','TI',101,6000,1),(4,'Leo','Ventas',102,8000,1),(5,'Sol','Ventas',102,5000,4),(6,'Paz','Ventas',102,4000,4);""")
q('SELECT COUNT(*) AS n,COUNT(manager_id) AS con_jefe,SUM(salary) AS total,AVG(salary) AS media,MAX(salary) AS maximo,MIN(salary) AS minimo FROM employees'); q('SELECT department,COUNT(*) AS n,AVG(salary) AS media FROM employees GROUP BY department HAVING COUNT(*)>=2')

,n,con_jefe,total,media,maximo,minimo
0,6,5,38000.0000,6333.3333,9000.0000,4000.0000


,department,n,media
0,TI,3,7000.0000
1,Ventas,3,5666.6667


## 2.12–2.13 Joins y alias

INNER devuelve coincidencias; LEFT conserva todas las filas izquierdas; RIGHT conserva las derechas; FULL OUTER conserva ambas. AS permite renombrar columnas y tablas. Un join puede multiplicar filas; verificar cardinalidad. Para demostrar no coincidencias de ambos lados se usan tablas independientes, sin violar claves foráneas.

In [20]:
run("CREATE TABLE a(id INTEGER PRIMARY KEY); CREATE TABLE b(id INTEGER PRIMARY KEY); INSERT INTO a VALUES(1),(2); INSERT INTO b VALUES(2),(3);")
for tipo in ['INNER','LEFT','RIGHT','FULL OUTER']:
    print(tipo)
    if tipo=='RIGHT': q('SELECT a.id AS a,b.id AS b FROM b LEFT JOIN a ON a.id=b.id')
    elif tipo=='FULL OUTER': q('SELECT a.id AS a,b.id AS b FROM a LEFT JOIN b ON a.id=b.id UNION ALL SELECT a.id,b.id FROM b LEFT JOIN a ON a.id=b.id WHERE a.id IS NULL')
    else: q(f'SELECT a.id AS a,b.id AS b FROM a {tipo} JOIN b ON a.id=b.id')
q('SELECT u.name AS cliente,o.order_id AS pedido FROM customers u INNER JOIN orders o ON u.id=o.customer_id')

INNER


,a,b
0,2,2


LEFT


,a,b
0,1,NaN
1,2,2.0000


RIGHT


,a,b
0,2.0000,2
1,NaN,3


FULL OUTER


,a,b
0,1.0000,NaN
1,2.0000,2.0000
2,NaN,3.0000


,cliente,pedido
0,Ana,1
1,Ana,2
2,Luis,3


## 2.14–2.15 Subconsultas y vistas

Una subconsulta es una consulta dentro de otra. Una vista es una tabla virtual definida por una consulta, no necesariamente materializada.

In [21]:
q('SELECT name FROM users WHERE id=(SELECT MAX(id) FROM users)'); run("CREATE VIEW active_users AS SELECT * FROM users WHERE status='active';"); q('SELECT name FROM active_users')

,name
0,Eva


,name
0,John Doe
1,Jane
2,Eva


## III. SQL avanzado

Consultas complejas, administración de transacciones y optimización.

## 3.1 Joins avanzados

SELF JOIN une una tabla consigo misma (empleado y gerente). CROSS JOIN produce el producto cartesiano. NATURAL JOIN une automáticamente por todas las columnas homónimas; se recomienda ON explícito para evitar cambios silenciosos al modificar el esquema.

In [22]:
q('SELECT e.name AS empleado,m.name AS gerente FROM employees e JOIN employees m ON e.manager_id=m.id'); run("CREATE TABLE categories(id INTEGER PRIMARY KEY,name TEXT); INSERT INTO categories VALUES(1,'A'),(2,'B');"); q('SELECT p.name AS producto,k.name AS categoria FROM products p CROSS JOIN categories k'); q('SELECT * FROM a NATURAL JOIN b')

,empleado,gerente
0,Ana,Eva
1,Luis,Eva
2,Leo,Eva
3,Sol,Leo
4,Paz,Leo


,producto,categoria
0,A,A
1,A,B
2,B,A
3,B,B


,id
0,2


## 3.2 Funciones de ventana

Calculan sobre filas relacionadas sin colapsarlas. ROW_NUMBER numera filas; RANK deja huecos en empates; DENSE_RANK no deja huecos; NTILE divide filas en grupos de tamaño parecido (puede separar empates); LAG y LEAD recuperan valores anteriores y posteriores. PARTITION BY reinicia el cálculo por grupo; ORDER BY define el orden.

In [23]:
q('SELECT name,department,salary,ROW_NUMBER() OVER(PARTITION BY department ORDER BY salary DESC,id) AS numero FROM employees')

,name,department,salary,numero
0,Eva,TI,9000.0000,1
1,Ana,TI,6000.0000,2
2,Luis,TI,6000.0000,3
3,Leo,Ventas,8000.0000,1
4,Sol,Ventas,5000.0000,2
5,Paz,Ventas,4000.0000,3


In [24]:
q('SELECT name,salary,RANK() OVER(ORDER BY salary DESC) AS rango,DENSE_RANK() OVER(ORDER BY salary DESC) AS rango_denso,NTILE(4) OVER(ORDER BY salary DESC,id) AS cuartil FROM employees')

,name,salary,rango,rango_denso,cuartil
0,Eva,9000.0000,1,1,1
1,Leo,8000.0000,2,2,1
2,Ana,6000.0000,3,3,2
3,Luis,6000.0000,3,3,2
4,Sol,5000.0000,5,4,3
5,Paz,4000.0000,6,5,4


In [25]:
q('SELECT name,salary,LAG(salary) OVER(ORDER BY salary,id) AS anterior,LEAD(salary) OVER(ORDER BY salary,id) AS siguiente FROM employees')

,name,salary,anterior,siguiente
0,Paz,4000.0000,NaN,5000.0000
1,Sol,5000.0000,4000.0000,6000.0000
2,Ana,6000.0000,5000.0000,6000.0000
3,Luis,6000.0000,6000.0000,8000.0000
4,Leo,8000.0000,6000.0000,9000.0000
5,Eva,9000.0000,8000.0000,NaN


## 3.3 CTE

WITH define un resultado temporal para una consulta. WITH RECURSIVE permite recorrer jerarquías, por ejemplo un organigrama. Añadimos profundidad y protección contra ciclos.

In [26]:
q('WITH EmployeeCTE AS (SELECT department,AVG(salary) AS avg_salary FROM employees GROUP BY department) SELECT * FROM EmployeeCTE')

,department,avg_salary
0,TI,7000.0000
1,Ventas,5666.6667


In [27]:
q("WITH RECURSIVE org(id,name,depth,path) AS (SELECT id,name,0,printf('/%d/',id) FROM employees WHERE manager_id IS NULL UNION ALL SELECT e.id,e.name,o.depth+1,o.path||e.id||'/' FROM employees e JOIN org o ON e.manager_id=o.id WHERE instr(o.path,printf('/%d/',e.id))=0) SELECT * FROM org ORDER BY depth,id")

,id,name,depth,path
0,1,Eva,0,/1/
1,2,Ana,1,/1/2/
2,3,Luis,1,/1/3/
3,4,Leo,1,/1/4/
4,5,Sol,2,/1/4/5/
5,6,Paz,2,/1/4/6/


## 3.4 Subconsultas avanzadas

Una subconsulta correlacionada depende de la consulta exterior. EXISTS comprueba si existe al menos una fila; no necesita recuperar sus columnas.

In [28]:
q('SELECT name,salary FROM employees e WHERE salary>(SELECT AVG(salary) FROM employees x WHERE x.department=e.department)')

,name,salary
0,Eva,9000.0000
1,Leo,8000.0000


In [29]:
q('SELECT name FROM customers c WHERE EXISTS(SELECT 1 FROM orders o WHERE o.customer_id=c.id)')

,name
0,Ana
1,Luis


## 3.5 Índices

Un índice acelera determinadas búsquedas mediante estructuras auxiliares. Puede ser simple o compuesto. CREATE INDEX y DROP INDEX lo crean y eliminan. Los índices ocupan espacio y encarecen escrituras; el orden de columnas importa.

In [30]:
run('CREATE INDEX idx_name ON employees(name); CREATE INDEX idx_dept_salary ON employees(department_id,salary);'); q('PRAGMA index_list(employees)'); q('EXPLAIN QUERY PLAN SELECT name FROM employees WHERE department_id=101 AND salary>=6000'); run('DROP INDEX idx_name;')

,seq,name,unique,origin,partial
0,0,idx_dept_salary,0,c,0
1,1,idx_name,0,c,0


,id,parent,notused,detail
0,3,0,0,SEARCH employees USING INDEX idx_dept_salary (...


## 3.6 Transacciones

Una transacción agrupa operaciones como unidad ACID. START TRANSACTION en MySQL equivale a BEGIN en SQLite. COMMIT confirma; ROLLBACK revierte; SAVEPOINT permite reversión parcial. Ejemplo original: transferir 100 entre dos cuentas. Se usan centavos enteros y se verifica conservación del saldo.

In [31]:
run('CREATE TABLE accounts(account_id INTEGER PRIMARY KEY,balance INTEGER NOT NULL CHECK(balance>=0)); INSERT INTO accounts VALUES(1,100000),(2,50000);')
def transferir(origen,destino,centavos):
    if centavos<=0 or origen==destino: raise ValueError('Transferencia inválida')
    with c:
        a=c.execute('UPDATE accounts SET balance=balance-? WHERE account_id=?',(centavos,origen))
        b=c.execute('UPDATE accounts SET balance=balance+? WHERE account_id=?',(centavos,destino))
        if a.rowcount!=1 or b.rowcount!=1: raise ValueError('Cuenta inexistente')
transferir(1,2,10000); q('SELECT account_id,balance/100.0 AS saldo FROM accounts')
try: transferir(1,2,999999)
except db.IntegrityError: print('Rollback por saldo insuficiente')
assert c.execute('SELECT SUM(balance) FROM accounts').fetchone()[0]==150000
c.execute('BEGIN'); c.execute('SAVEPOINT s'); c.execute('UPDATE accounts SET balance=0 WHERE account_id=1'); c.execute('ROLLBACK TO s'); c.execute('RELEASE s'); c.commit(); q('SELECT * FROM accounts')

,account_id,saldo
0,1,900.0000
1,2,600.0000


Rollback por saldo insuficiente


,account_id,balance
0,1,90000
1,2,60000


## 3.7 Procedimientos almacenados

Código SQL reutilizable almacenado en el servidor. Puede reducir tráfico y facilitar permisos; no garantiza mayor velocidad. MySQL: DELIMITER //; CREATE PROCEDURE GetEmployeesByDept(IN dept_id INT) BEGIN SELECT * FROM employees WHERE department_id=dept_id; END //; DELIMITER ; y CALL GetEmployeesByDept(101). DELIMITER es una orden del cliente, no SQL del servidor. SQLite carece de procedimientos almacenados; la función Python es una alternativa del lado cliente.

In [32]:
def empleados_por_departamento(dept_id):
    return pd.read_sql_query('SELECT * FROM employees WHERE department_id=?',c,params=(dept_id,))
display(empleados_por_departamento(101))

,id,name,department,department_id,salary,manager_id
0,2,Ana,TI,101,6000.0000,1.0000
1,3,Luis,TI,101,6000.0000,1.0000
2,1,Eva,TI,101,9000.0000,NaN


## 3.8 Triggers

Acciones automáticas ante INSERT, UPDATE o DELETE. El ejemplo MySQL BEFORE INSERT asigna SET NEW.created_at = NOW(). En SQLite se usa DEFAULT CURRENT_TIMESTAMP para la fecha y un trigger AFTER INSERT para auditoría. Los triggers deben documentarse por sus efectos indirectos.

In [33]:
run("""CREATE TABLE audit(id INTEGER PRIMARY KEY,user_id INTEGER,event TEXT,created_at TEXT DEFAULT CURRENT_TIMESTAMP); CREATE TRIGGER audit_insert AFTER INSERT ON users BEGIN INSERT INTO audit(user_id,event) VALUES(NEW.id,'INSERT'); END; INSERT INTO users(name,email) VALUES('Sol','sol@example.com');"""); q('SELECT * FROM audit')

,id,user_id,event,created_at
0,1,4,INSERT,2026-11-27 02:32:23


## 3.9 Particionamiento

Divide tablas grandes por rango, hash o lista. Ejemplo MySQL: CREATE TABLE sales(id INT, amount DECIMAL(12,2), sale_date DATE) PARTITION BY RANGE(YEAR(sale_date)) (PARTITION p1 VALUES LESS THAN (2020), PARTITION p2 VALUES LESS THAN (2025), PARTITION pmax VALUES LESS THAN MAXVALUE); se añade MAXVALUE para fechas posteriores. SQLite no ofrece particionamiento nativo: se simula la distribución de filas; no equivale a optimización física.

In [34]:
sales=pd.DataFrame({'id':[1,2,3,4],'amount':[10,20,30,40],'sale_date':['2019-01-01','2021-01-01','2025-01-01','2026-01-01'],'region':['N','S','N','O']}); year=pd.to_datetime(sales.sale_date).dt.year
sales['rango']=pd.cut(year,[-float('inf'),2019,2024,float('inf')],labels=['p1','p2','pmax']); sales['hash']=sales.id%2; sales['lista']=sales.region.map({'N':'norte','S':'sur','O':'otros'}); display(sales); display(sales.groupby('rango',observed=True).amount.sum())

,id,amount,sale_date,region,rango,hash,lista
0,1,10,2019-01-01,N,p1,1,norte
1,2,20,2021-01-01,S,p2,0,sur
2,3,30,2025-01-01,N,pmax,1,norte
3,4,40,2026-01-01,O,pmax,0,otros


rango
p1      10
p2      20
pmax    70
Name: amount, dtype: int64

## 3.10 Optimización avanzada

EXPLAIN analiza el plan; SQLite ofrece EXPLAIN QUERY PLAN. Indexar columnas frecuentes, seleccionar solo las necesarias y limitar resultados. El original recomienda joins frente a subconsultas cuando sea posible; no siempre son más rápidos ni equivalentes. Medir el plan y verificar igualdad de resultados.

In [35]:
run('CREATE TABLE big(id INTEGER PRIMARY KEY,dept INTEGER,amount INTEGER);'); c.executemany('INSERT INTO big VALUES(?,?,?)',[(i,i%100,i*3) for i in range(20000)]); c.commit()
sql='SELECT id,amount FROM big WHERE dept=42 ORDER BY id LIMIT 10'; before=pd.read_sql_query(sql,c); q('EXPLAIN QUERY PLAN '+sql); run('CREATE INDEX idx_big_dept ON big(dept);'); q('EXPLAIN QUERY PLAN '+sql); after=pd.read_sql_query(sql,c); pd.testing.assert_frame_equal(before,after); display(after)
assert c.execute('PRAGMA integrity_check').fetchone()[0]=='ok'; assert not c.execute('PRAGMA foreign_key_check').fetchall()

,id,parent,notused,detail
0,4,0,0,SCAN big


,id,parent,notused,detail
0,5,0,0,SEARCH big USING INDEX idx_big_dept (dept=?)


,id,amount
0,42,126
1,142,426
2,242,726
3,342,1026
4,442,1326
5,542,1626
6,642,1926
7,742,2226
8,842,2526
9,942,2826
